# Stage 1: vLLM serving on a free Colab T4

Serves Qwen2.5-0.5B-Instruct with vLLM and loads it with `vllm bench serve` from the same machine:
512 random input tokens, 128 output tokens, concurrency 1, 4, 16, 32, three repeats in one session.
Run the cells top to bottom (Runtime > Change runtime type > T4 GPU). The last cell prints every result file;
save the notebook with its outputs and run `bench/extract_results.py` and `bench/summarize.py` on it
(see `docs/stage1-serving.md`).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!python --version

In [ ]:
!pip install -q vllm --extra-index-url https://download.pytorch.org/whl/cu129
!pip list 2>/dev/null | grep -i -E "^(vllm|torch|transformers) "
# the flags the benchmark cell passes must all be listed here
!vllm bench serve --help=all 2>&1 | grep -o -E -- "--(backend|base-url|dataset-name|ignore-eos|seed|result-dir|result-filename|metric-percentiles|max-concurrency|random-input-len|random-output-len|num-prompts|save-result)\b" | sort -u

In [ ]:
import subprocess, time, urllib.request

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
# --dtype half: the model ships as bfloat16 and the T4 (compute capability 7.5) has no bfloat16
server = subprocess.Popen(
    ["vllm", "serve", MODEL, "--dtype", "half", "--max-model-len", "4096"],
    stdout=open("server.log", "w"), stderr=subprocess.STDOUT)
for _ in range(90):
    try:
        urllib.request.urlopen("http://localhost:8000/v1/models", timeout=2)
        break
    except OSError:
        if server.poll() is not None:
            break
        time.sleep(10)
!tail -n 12 server.log
assert server.poll() is None, "vllm serve exited, see server.log"
urllib.request.urlopen("http://localhost:8000/v1/models", timeout=2)
print("server ready")

In [ ]:
import itertools, json, os

os.makedirs("results", exist_ok=True)

def bench(name, concurrency, prompts, seed):
    out = f"results/{name}.json"
    if not os.path.exists(out):  # a cut session resumes at the first missing file
        with open(f"results/{name}.log", "w") as log:
            subprocess.run(
                ["vllm", "bench", "serve", "--backend", "openai", "--base-url", "http://localhost:8000",
                 "--model", MODEL, "--dataset-name", "random",
                 "--random-input-len", "512", "--random-output-len", "128", "--ignore-eos",
                 "--num-prompts", str(prompts), "--max-concurrency", str(concurrency),
                 # a new seed per run: identical prompts would hit the server's prefix cache
                 "--seed", str(seed), "--metric-percentiles", "50,95",
                 "--save-result", "--result-dir", "results", "--result-filename", f"{name}.json"],
                stdout=log, stderr=subprocess.STDOUT, check=True)
    r = json.load(open(out))
    print(name, {k: round(v, 1) for k, v in r.items()
                 if k in ("completed", "duration", "output_throughput", "median_ttft_ms", "p95_ttft_ms")})

bench("warmup", 4, 64, 1)  # discarded
for repeat, concurrency in itertools.product((1, 2, 3), (1, 4, 16, 32)):
    bench(f"c{concurrency:02d}_r{repeat}", concurrency, 256, 100 * repeat + concurrency)

In [ ]:
import glob

# the server's own first-token histogram, to cross-check the client numbers above
!curl -s localhost:8000/metrics | grep -E "^vllm:time_to_first_token_seconds_(sum|count)"
print("<<<RESULTS")
for path in sorted(glob.glob("results/*.json")):
    print(json.dumps({"file": os.path.basename(path), **json.load(open(path))}))
print("RESULTS>>>")